# COCO Evaluation 101

This notebook walks through COCO object detection evaluation with **hotcoco** — a drop-in replacement for pycocotools that runs 11–26x faster.

**What you'll learn:**
- Run standard COCO evaluation (AP, AR) on your detections
- Interpret the 12 standard metrics
- Get per-class AP breakdown
- Log metrics to Weights & Biases / MLflow
- Diagnose errors with TIDE decomposition
- Drop in as a replacement for any pycocotools-based training pipeline

---

## Install

In [ ]:
%pip install hotcoco

## 1. Quickstart — synthetic data

No dataset download required. We'll build a tiny ground truth + detections dict in memory to verify everything works.

In [ ]:
from hotcoco import COCO, COCOeval

# Minimal in-memory COCO dataset
dataset = {
    "images": [{"id": 1, "width": 640, "height": 480}],
    "categories": [{"id": 1, "name": "cat"}, {"id": 2, "name": "dog"}],
    "annotations": [
        {"id": 1, "image_id": 1, "category_id": 1, "bbox": [10, 20, 100, 80],  "area": 8000,  "iscrowd": 0},
        {"id": 2, "image_id": 1, "category_id": 2, "bbox": [200, 150, 120, 90], "area": 10800, "iscrowd": 0},
    ],
}

detections = [
    {"image_id": 1, "category_id": 1, "bbox": [12, 22, 98, 78],  "score": 0.95},
    {"image_id": 1, "category_id": 2, "bbox": [205, 155, 115, 85], "score": 0.80},
    {"image_id": 1, "category_id": 1, "bbox": [300, 300, 50, 50], "score": 0.30},  # FP
]

coco_gt = COCO(dataset)
coco_dt = coco_gt.load_res(detections)

ev = COCOeval(coco_gt, coco_dt, "bbox")
ev.evaluate()
ev.accumulate()
ev.summarize()

## 2. Full COCO val2017 evaluation

Download COCO val2017 annotations and a detections file, then swap in the paths below.

```bash
# Download annotations (if you don't have them)
wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip
unzip annotations_trainval2017.zip
```

Any model that outputs COCO-format JSON detections works here.

In [ ]:
GT_PATH = "annotations/instances_val2017.json"
DT_PATH = "detections.json"  # your model's output

coco_gt = COCO(GT_PATH)
coco_dt = coco_gt.load_res(DT_PATH)

ev = COCOeval(coco_gt, coco_dt, "bbox")
ev.run()  # shorthand for evaluate() + accumulate() + summarize()

### Understanding the 12 metrics

| Metric | Description |
|--------|-------------|
| **AP** | mAP averaged over IoU thresholds 0.50:0.05:0.95 |
| **AP50** | mAP at IoU ≥ 0.50 (PASCAL VOC style) |
| **AP75** | mAP at IoU ≥ 0.75 (strict) |
| **APs / APm / APl** | AP for small / medium / large objects |
| **AR1 / AR10 / AR100** | Max recall given 1, 10, 100 detections per image |
| **ARs / ARm / ARl** | AR for small / medium / large objects |

Small = area < 32², medium = 32²–96², large = area > 96².

In [ ]:
# Get results as a dict — useful for logging or further analysis
results = ev.get_results()
print(results)
# {'AP': 0.382, 'AP50': 0.584, 'AP75': 0.412, ...}

## 3. Per-class AP

`per_class=True` adds one entry per category, keyed as `AP/{name}`.

In [ ]:
results = ev.get_results(per_class=True)

# Pull out per-class AP and sort descending
per_class = {k: v for k, v in results.items() if k.startswith("AP/")}
for cat, ap in sorted(per_class.items(), key=lambda x: -x[1]):
    print(f"  {cat:<30} {ap:.3f}")

## 4. Logging to experiment trackers

`prefix` prepends a path to all metric keys so they group cleanly in your tracker's UI.

In [ ]:
# Weights & Biases
import wandb
wandb.log(ev.get_results(prefix="val/bbox", per_class=True), step=epoch)
# Logs: {"val/bbox/AP": 0.382, "val/bbox/AP/person": 0.82, ...}

In [ ]:
# MLflow
import mlflow
mlflow.log_metrics(ev.get_results(prefix="val/bbox"), step=epoch)

## 5. F-scores

`f_scores()` finds the confidence threshold that maximises F-beta for each (IoU threshold, category), then averages — analogous to mAP but optimising the F-beta operating point.

Useful when you care about precision/recall balance at deployment, not just the area under the curve.

In [ ]:
ev.f_scores()          # F1 (balanced)
# {'F1': 0.523, 'F150': 0.712, 'F175': 0.581}

ev.f_scores(beta=0.5)  # favour precision
ev.f_scores(beta=2.0)  # favour recall

## 6. TIDE error analysis

TIDE decomposes every false positive and false negative into six error types:

| Error | Meaning |
|-------|---------|
| **Loc** | Correct class, box IoU too low |
| **Cls** | Correct box, wrong class |
| **Dupe** | Duplicate detection of an already-matched GT |
| **Bkg** | Detection with no nearby GT (pure background) |
| **Both** | Wrong class *and* wrong box |
| **Miss** | GT with no detection attempting to match it |

ΔAP = how much AP would improve if that error type were fixed.

In [ ]:
ev = COCOeval(coco_gt, coco_dt, "bbox")
ev.evaluate()

tide = ev.tide_errors()

print(f"Base AP: {tide['ap_base']:.4f}")
print()
print("Error breakdown (sorted by impact):")
error_types = ["Loc", "Cls", "Dupe", "Bkg", "Both", "Miss"]
for name in sorted(error_types, key=lambda k: -tide["delta_ap"].get(k, 0)):
    delta = tide["delta_ap"].get(name, 0)
    count = tide["counts"].get(name, 0)
    print(f"  {name:<8} ΔAP={delta:.4f}  n={count}")

**Reading the output:** the largest ΔAP tells you where to focus.
- High **Loc** → improve bounding box regression
- High **Cls** → improve classification head or NMS
- High **Miss** → model misses objects entirely; try lower confidence threshold or data augmentation
- High **Bkg** → too many false positives on background; tighten confidence threshold

## 7. Drop-in replacement for pycocotools

If you use Detectron2, Ultralytics YOLO, mmdetection, or any other pipeline that imports `pycocotools`, one call at startup replaces everything with hotcoco — no other changes needed.

In [ ]:
from hotcoco import init_as_pycocotools
init_as_pycocotools()

# These now resolve to hotcoco under the hood
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask

## 8. Segmentation and keypoint evaluation

The API is identical — just swap `"bbox"` for `"segm"` or `"keypoints"`.

In [ ]:
# Segmentation
coco_gt = COCO("annotations/instances_val2017.json")
coco_dt = coco_gt.load_res("segm_results.json")
ev = COCOeval(coco_gt, coco_dt, "segm")
ev.run()

# Keypoints
coco_gt = COCO("annotations/person_keypoints_val2017.json")
coco_dt = coco_gt.load_res("kpt_results.json")
ev = COCOeval(coco_gt, coco_dt, "keypoints")
ev.run()

## 9. Dataset inspection

`COCO` is also a full dataset API — you can query images, annotations, and categories.

In [ ]:
coco = COCO("annotations/instances_val2017.json")

# List all category names
cats = coco.load_cats(coco.get_cat_ids())
print([c["name"] for c in cats])

# Get all annotation IDs for a given image + category
ann_ids = coco.get_ann_ids(img_ids=[139], cat_ids=[1])
anns = coco.load_anns(ann_ids)
print(anns)

---

## Next steps

- [Full API reference](https://derekallman.github.io/hotcoco/api/cocoeval/)
- [LVIS federated evaluation](https://derekallman.github.io/hotcoco/user-guide/evaluation/#lvis)
- [CLI reference](https://derekallman.github.io/hotcoco/cli/)
- [GitHub](https://github.com/derekallman/hotcoco)